### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="amex_non_iid",
    dataset_year="2022",
    domain_str="finance",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/competitions/amex-default-prediction/",
    download_description="""
We start with the preprocessed version of the dataset in Parquet format provided by
the user 'raddar' on Kaggle. At the root of this project, I ran on linux with
the kaggle CLI installed and authenticated:

kaggle datasets download -d raddar/amex-data-integer-dtypes-parquet-format --unzip -p amex_dataset && rm amex_dataset/test.parquet
kaggle competitions download -c amex-default-prediction -f train_labels.csv -p amex_dataset && cd amex_dataset && unzip train_labels.csv.zip train_labels.csv && rm train_labels.csv.zip && cd ..
mkdir -p local-data-warehouse/amex_non_iid && mv amex_dataset/* local-data-warehouse/amex_non_iid/ && rm -rf amex_dataset
""",
    # References
    academic_reference_bibtex="""@misc{howard2022amex,
  author       = {Howard, Addison and AritraAmex and Xu, Di and Vashani, Hossein and inversion and Negin and Dane, Sohier},
  title        = {American Express -- Default Prediction},
  year         = {2022},
  howpublished = {Kaggle Competition},
  url          = {https://kaggle.com/competitions/amex-default-prediction},
  note         = {Accessed via Kaggle}
}
""",
    academic_reference_bibtex_key="howard2022amex",
    licence="Kaggle Competition License",
    data_tags=["Non-IID", "Temporal", "Grouped", "Anonymized"],
    curation_comments="""
We start with the raw data from Kaggle and do not apply any further preprocessing to simulate a pipeline that can handle raw non-IID grouped data.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="target",
    problem_type="binary_classification",
    objective_metric_name="amex_metric",
    stratify_on="target",
    group_on="customer_ID",
    time_on="S_2",
)

## Preprocessing

In [2]:
import pandas as pd

df = pd.read_parquet(dataset_mold.path / "train.parquet")
print("Loaded data shape:", df.shape)

df.S_2 = pd.to_datetime(df.S_2)
df = df.sort_values(["customer_ID", "S_2"])

cat_features = [
    "B_30",
    "B_38",
    "D_114",
    "D_116",
    "D_117",
    "D_120",
    "D_126",
    "D_63",
    "D_64",
    "D_66",
    "D_68",
]
df = df.set_index("customer_ID")
# Add labels
targets = pd.read_csv(dataset_mold.path / "train_labels.csv")
targets = targets.set_index("customer_ID")
df = df.merge(targets, left_index=True, right_index=True, how="left")
df.target = df.target.astype("int8")
del targets

df[cat_features] = df[cat_features].astype("category")

df = df.reset_index().reset_index(drop=True)

Loaded data shape: (5531451, 190)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False,
    duplicate_column_check=False,
)


#### Dataset Overview
Rows: 5,531,451
Columns: 191

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,customer_ID,S_2,P_2,D_39,B_1,B_2,R_1,S_3,D_41,B_3,D_42,D_43,D_44,B_4,D_45,B_5,R_2,D_46,D_47,D_48,D_49,B_6,B_7,B_8,D_50,D_51,B_9,R_3,D_52,P_3,B_10,D_53,S_5,B_11,S_6,D_54,R_4,S_7,B_12,S_8,D_55,D_56,B_13,R_5,D_58,S_9,B_14,D_59,D_60,D_61,B_15,S_11,D_62,D_63,D_64,D_65,B_16,B_17,B_18,B_19,D_66,B_20,D_68,S_12,R_6,S_13,B_21,D_69,B_22,D_70,D_71,D_72,S_15,B_23,D_73,P_4,D_74,D_75,D_76,B_24,R_7,D_77,B_25,B_26,D_78,D_79,R_8,R_9,S_16,D_80,R_10,R_11,B_27,D_81,D_82,S_17,R_12,B_28,R_13,D_83,R_14,R_15,D_84,R_16,B_29,B_30,S_18,D_86,D_87,R_17,R_18,D_88,B_31,S_19,R_19,B_32,S_20,R_20,R_21,B_33,D_89,R_22,R_23,D_91,D_92,D_93,D_94,R_24,R_25,D_96,S_22,S_23,S_24,S_25,S_26,D_102,D_103,D_104,D_105,D_106,D_107,B_36,B_37,R_26,R_27,B_38,D_108,D_109,D_110,D_111,B_39,D_112,B_40,S_27,D_113,D_114,D_115,D_116,D_117,D_118,D_119,D_120,D_121,D_122,D_123,D_124,D_125,D_126,D_127,D_128,D_129,B_41,B_42,D_130,D_131,D_132,D_133,R_28,D_134,D_135,D_136,D_137,D_138,D_139,D_140,D_141,D_142,D_143,D_144,D_145,target
0,0000099d6bd597052cdcda90ffabf56573fe9d7c79be5fbac11a8ed792feb62a,2017-03-09,0.938469,0,0.008724,1.006838,0.009228,0.124035,0.0,0.004709,NaN,NaN,0,6,0.708906,0.170600,0,0.358587,0.525351,0.255736,-1,0.063902,0.059416,0.0,0.148698,4,0.008207,0,0.207334,0.736463,0.096219,NaN,0.023381,0.002768,0,1.0,0,0.161345,0.148266,2896,0.354596,0.152025,0.118075,0,0.158612,0.065728,0.018385,8,0.199617,0.308233,0.016361,15,0.091071,0,0,0,0,NaN,0.652984,0,-1,0,6,0.272008,0.008363,524,0.002644,0.009013,0,0,0.119403,0,4,0.050882,NaN,0.0,1,1,NaN,0.004327,0.0,NaN,0.007729,0.000272,0,0,0,-1,0.002271,0,0,0,0.002310,0,1,0.008033,1.0,0.084683,0,0,0.0,0,0,0,NaN,0,0,0,-1,0,0,NaN,1,0.002537,0,0,0,0,0,1,0,0,0,3,1,0,0,0,0,0,0.894090,0.135561,0.911191,0.974539,0.001243,0.766688,1,1.004587,0.893734,-1,2,0.009968,0.004572,-1,1.008949,2,-1,0,NaN,-1,NaN,1.0,0.210060,0.676922,0,1,0.238250,0,5,0.232120,0.236266,0,0.702280,3,0,16,0,2,1,1.007819,1,0,NaN,0.0,0.0,NaN,0.004345,0,NaN,-1,-1,-1,-1,0,0,0.0,NaN,0,0.000610,0,0
1,0000099d6bd597052cdcda90ffabf56573fe9d7c79be5fbac11a8ed792feb62a,2017-04-07,0.936665,0,0.004923,1.000653,0.006151,0.126750,0.0,0.002714,NaN,NaN,0,5,0.712795,0.113239,0,0.353630,0.521311,0.223329,-1,0.065261,0.057744,0.0,0.149723,4,0.008373,0,0.202778,0.720886,0.099804,NaN,0.030599,0.002749,0,1.0,0,0.140951,0.143530,2896,0.326757,0.156201,0.118737,0,0.148459,0.093935,0.013035,8,0.151387,0.265026,0.017688,15,0.086805,0,0,0,0,NaN,0.647093,0,-1,0,6,0.188970,0.004030,524,0.004193,0.007842,0,0,0.140611,0,4,0.040469,NaN,0.0,1,1,NaN,0.004203,0.0,NaN,0.001864,0.000979,0,0,0,-1,0.009810,0,0,0,0.001327,0,1,0.000760,1.0,0.081843,0,0,0.0,0,0,0,NaN,0,0,0,-1,0,0,NaN,1,0.008427,0,0,0,0,0,1,0,0,0,3,1,0,0,0,0,0,0.902135,0.136333,0.919876,0.975625,0.004561,0.786007,1,1.004118,0.906841,-1,2,0.003921,0.004654,-1,1.003205,2,-1,0,NaN,-1,NaN,1.0,0.184093,0.822281,0,1,0.247217,0,5,0.243532,0.241885,0,0.707017,3,0,16,0,2,1,1.004333,1,0,NaN,0.0,0.0,NaN,0.007495,0,NaN,-1,-1,-1,-1,0,0,0.0,NaN,0,0.005492,0,0
2,0000099d6bd597052cdcda90ffabf56573fe9d7c79be5fbac11a8ed792feb62a,2017-05-28,0.954180,3,0.021655,1.009672,0.006815,0.123977,0.0,0.009423,NaN,NaN,0,5,0.720884,0.060492,0,0.334650,0.524568,0.189424,-1,0.066982,0.056647,0.0,0.151955,4,0.009355,0,0.206629,0.738044,0.134073,NaN,0.048367,0.010077,0,1.0,0,0.112229,0.137014,3166,0.304124,0.153795,0.114534,0,0.139504,0.084757,0.056653,8,0.305883,0.212165,0.063955,15,0.094001,0,0,0,0,NaN,0.645819,0,-1,0,6,0.495308,0.006838,702,0.001337,0.006025,0,0,0.075868,0,4,0.047454,NaN,0.0,1,1,NaN,0.001782,0.0,NaN,0.005419,0.006149,0,0,0,-1,0.009362,0,0,0,0.007624,0,1,0.004056,1.0,0.081954,0,0,0.0,0,0,0,NaN,0,0,0,-1,0,0,NaN,1,0.007327,0,0,0,0,0,1,0,0,0,3,1,0,0,0,0,0,0.939654,0.134938,0.958699,0.974067,0.011736,0.806840,1,1.009285,0.928719,-1,2,0.001264,0.019176,-1,1.000754,2,-1,0,NaN,-1,NaN,1.0,0.154837,0.853498,0,1,0.239867,0,5,0.240768,0.239710,0,0.704843,3,0,16,0,2,1,1.007831,1,0,NaN,0.0,0.0,NaN,0.009227,0,NaN,-1,-1,-1,-1,0,0,0.0,NaN,0,0.006986,0,0
3,0000099d6bd597052cdcda90ffabf

In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,D_63,category,0,0.00,6,"3, 0, 4, 1, 2, 5"
1,D_64,category,0,0.00,5,"0, 3, 2, -1, 1"
2,D_66,category,0,0.00,3,"-1, 1, 0"
3,D_68,category,0,0.00,8,"6, 5, 3, 4, 2, -1, 1, 0"
4,B_30,category,0,0.00,4,"0, 1, 2, -1"
5,B_38,category,0,0.00,8,"2, 3, 1, 5, 4, 7, 6, -1"
6,D_114,category,0,0.00,3,"1, 0, -1"
7,D_116,category,0,0.00,3,"0, -1, 1"
8,D_117,category,0,0.00,8,"0, 4, 5, 3, 6, 7, -1, 2"
9,D_120,category,0,0.00,3,"0, 1, -1"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
P_2,5485466.0,0.656334,0.244649,-4.589548e-01,1.010000
D_39,5531451.0,5.035986,9.181833,0.000000e+00,183.000000
B_1,5531451.0,0.124010,0.211987,-7.588799e+00,1.324060
B_2,5529435.0,0.621489,0.401488,9.192280e-09,1.010000
R_1,5531451.0,0.078803,0.226397,1.534223e-09,3.256284
S_3,4510907.0,0.225845,0.193347,-6.271321e-01,5.482888
D_41,5529435.0,0.055435,0.203707,0.000000e+00,8.988807
B_3,5529435.0,0.132539,0.234993,6.285293e-09,1.625262
D_42,791314.0,0.184974,0.228185,-4.543303e-04,4.191119
D_43,3873055.0,0.154684,0.213398,1.154550e-07,10.111619


In [7]:
# Categorical Feature Statistics
cat_stats

value  \
column      rank                                                                     
B_30        1                                                                    0   
            2                                                                    1   
            3                                                                    2   
            4                                                                   -1   
B_38        1                                                                    2   
            2                                                                    3   
            3                                                                    1   
            4                                                                    5   
            5                                                                    4   
D_114       1                                                                    1   
            2                                                                    0   
            3                                                                   -1   
D_116       1                                                                    0   
            2                                                                   -1   
            3                                                                    1   
D_117       1                                                                    0   
            2                                                                    4   
            3                                                                    5   
            4                                                                    3   
            5                                                                    6   
D_120       1                                                                    0   
            2                                                                    1   
            3                                                                   -1   
D_126       1                                                                    2   
            2                                                                    1   
            3                                                                    0   
            4                                                                   -1   
D_63        1                                                                    3   
            2                                                                    0   
            3                                                                    4   
            4                                                                    1   
            5                                                                    2   
D_64        1                                                                    0   
            2                                                                    3   
            3                                                                    2   
            4                                                                   -1   
            5                                                                    1   
D_66        1                                                                   -1   
            2                                                                    1   
            3                                                                    0   
D_68        1                                                                    6   
            2                                                                    5   
            3                                                                    3   
            4                                                                    4   
            5                                                                    2   
S_2         1                      

In [8]:
# Target Distribution
target_df

,count,pct
target,,
0,4153582,75.09
1,1377869,24.91


## Task Curation

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from sklearn.model_selection import StratifiedGroupKFold

splits = {0: {}}

sklearn_splits = StratifiedGroupKFold(n_splits=3, random_state=42, shuffle=True).split(
    X=df,
    y=df[task_mold.target_column_name],
    groups=df[task_mold.group_on],
)
print_once = False
for fold_idx, (train_index, test_index) in enumerate(sklearn_splits):
    # Print len, target col count, and group counts
    train_data = df.iloc[train_index]
    test_data = df.iloc[test_index]

    if not print_once:
        print(f"""Train N: {len(train_index)}, Test N: {len(test_index)}
        Target Distribution:
        \tTrain target distribution: {df.iloc[train_index][task_mold.target_column_name].value_counts(normalize=True).to_dict()}
        \tTest target distribution: {df.iloc[test_index][task_mold.target_column_name].value_counts(normalize=True).to_dict()}
        Group Distribution {task_mold.group_on}:
        \tTrain: {len(train_data[task_mold.group_on].unique())}
        \tTest: {len(test_data[task_mold.group_on].unique())}
        """
        )
        print_once = True
    splits[0][fold_idx] = (train_index.tolist(), test_index.tolist())

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="We create stratified grouped 3-fold split. This represents 150k (33%) customers and 1.8 million samples per test split. The label are equally represented in all train and test sets. Note, we will likely only use the first fold.",
        splits=splits
)

## Export

In [10]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
019c392e-d78c-7969-9f83-1e3bb2eb8a21
8cb9c9d3873cdc11fbefe37bffa09e02f39182ce24e0e2b0b464afd42d1f97cf
